# RQ3 rewriter fingerprint on the full corpus (Colab)

Runs `scripts/rq3_full.py`: Samer's paraphraser-identification protocol on all 7 datasets. It covers about 200k ChatGPT / PaLM / Dipper / Pegasus rewrites (T1-T3) of 16,682 originals, with per-dataset article-grouped CV. There are four experiments: main (5 models), cross-round, leave-one-source-out and typography, which makes 189 jobs. Protocol: `docs/protocol_rq3.md`.

It is **CPU-only**. Use the same runtime as RQ2 (**A100**, for its ~12 vCPUs; the CPU runtime has only 2).

**Before running**
1. The RQ2 full-corpus run must be finished in `MyDrive/sot_colab/derived/`. This run reuses its `rq2_full_texts.parquet` and `rq2_full_features.parts/full_text_*.parquet`, so there is no new spaCy pass.
2. On the Mac, rebuild the code bundle (from `Sam/`) and upload it to **`MyDrive/sot_colab/`**, replacing the old one:
   ```bash
   cd Sam && zip -r -9 data/colab/sot_rq2_bundle.zip src scripts data/derived/sot_style_vocab.json -x '*__pycache__*'
   ```
3. Choose `REPEATS` in cell 1: **1** (recommended, ~1 h) or **5** (Samer's full protocol, ~2.5-3 h). The two settings write to separate part folders.
4. Run all cells.

**Disconnects:** reconnect and run all cells again. Finished jobs are skipped.

**Output:** download `MyDrive/sot_colab/derived/rq3_full_predictions.parquet` and `rq3_full_pool_features.parquet` into `Sam/data/derived/`. Neither file contains any text.

In [ ]:
# 1. Drive, unpack the code bundle, check the RQ2 inputs
import os, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPEATS = 1   # 1 (recommended) or 5 (Samer's full protocol)

DRIVE = Path("/content/drive/MyDrive/sot_colab")
DERIVED, WORK = DRIVE / "derived", Path("/content/work")
assert (DRIVE / "sot_rq2_bundle.zip").exists(), f"upload sot_rq2_bundle.zip to {DRIVE} first"
assert (DERIVED / "rq2_full_texts.parquet").exists(), "run the RQ2 full-corpus notebook first"
assert len(list((DERIVED / "rq2_full_features.parts").glob("full_text_*.parquet"))), "RQ2 features missing"
print(f"vCPUs: {os.cpu_count()} | RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.0f} GB")

shutil.unpack_archive(DRIVE / "sot_rq2_bundle.zip", WORK)   # always: newest code
assert (WORK / "scripts/rq3_full.py").exists(), "old bundle: rebuild it with scripts/rq3_full.py"
ENV = {**os.environ, "PYTHONPATH": str(WORK / "src"), "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1",
       "MKL_NUM_THREADS": "1"}   # one thread per worker process


def run(*args, log="rq3_full.log"):
    """Run scripts/rq3_full.py against the Drive caches; stream progress lines and keep the full log on Drive."""
    cmd = [sys.executable, "scripts/rq3_full.py", *args, "--derived", str(DERIVED), "--repeats", str(REPEATS),
           "--workers", str(os.cpu_count())]
    print(" ".join(cmd[1:]), flush=True)
    t = time.time()
    with open(DERIVED / log, "a") as f, subprocess.Popen(
            cmd, cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if not line.startswith("cv ") or "tfidf_char" in line and "main" in line:
                print(line, end="", flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f"rq3_full.py failed (exit {p.returncode}); see {DERIVED / log}")
    print(f"-> finished in {(time.time() - t) / 60:.1f} min\n", flush=True)


print("ready")

In [ ]:
# 2. Cross-validation (resumable), then a completeness check
import pandas as pd

run()

preds = pd.read_parquet(DERIVED / "rq3_full_predictions.parquet",
                        columns=["dataset", "experiment", "model", "train_stage", "held_out_source"])
jobs = preds.groupby(["dataset", "experiment", "model", "train_stage", "held_out_source"], observed=True,
                     dropna=False).size()
desc = pd.read_parquet(DERIVED / "rq3_full_pool_features.parquet", columns=["stage"])
print(f"{len(preds):,} predictions | {len(jobs)} / 189 jobs | pool texts: {(desc.stage != 'T0').sum():,}")
assert len(jobs) == 189, "incomplete: run all cells again to resume"
print(f"Done. Download rq3_full_predictions.parquet and rq3_full_pool_features.parquet from {DERIVED} "
      "into the repo's Sam/data/derived/.")